In [2]:
import numpy as np
import pandas as pd

# KNN (K-Nearest-NEighbors)

KNN is a simple concept: define some distance metric between the items in your dataset, and find the K closet items. You can then use those items to predict some porperty of a test item, by having them somehoew "vote" on it.

As an example, lets look at the movielensa data. well try to guess the reting of a movie by looking at the 10 movies that are closet to it in terms of genres and popularity.

To start, wel load up every rating in the data set into pandas dataframe:

In [3]:
r_cols = ['user_id', 'movie_id', 'rating'] #define as colunas
ratings = pd.read_csv('anexos/Aquivos_de_C%C3%B3digo/u.data', sep='\t', names=r_cols, usecols=range(3)) #le o csv e coloca as colunas
ratings.head()

,user_id,movie_id,rating
0,0,50,5
1,0,172,5
2,0,133,1
3,196,242,3
4,186,302,3


Now, well group everything by movie ID, and compute the total number of ratings (each movie popularity) and the average rating for every movie:

In [4]:
movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]}) #agrupa os dados por id e faz o tanto de avaliacoes e a media delas
movieProperties.head()

/tmp/ipykernel_21412/1140431177.py:1: FutureWarning: The provided callable <function mean at 0x7fd4546e35e0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]}) #agrupa os dados por id e faz o tanto de avaliacoes e a media delas


rating          
           size      mean
movie_id                 
1           452  3.878319
2           131  3.206107
3            90  3.033333
4           209  3.550239
5            86  3.302326

In [5]:
movieNumRatings = pd.DataFrame(movieProperties['rating']['size']) #faz um dataframe acessando o rating e o size do movieproperties
movieNormalizedNumRatings = movieNumRatings.apply(lambda x: (x - np.min(x)) / (np.max(x) - np.min(x))) #normaliza os numeros 
movieNormalizedNumRatings.head()

,size
movie_id,
1,0.773585
2,0.222985
3,0.152659
4,0.356775
5,0.145798


Now lets get the genre information from the u.tem file. The way this works if there are 19 fields, each corresponding to a specific genre - a calue of '0' means it is not in that genre, and '1' means its in that genre. A movie way have ore than one genre associated with it.

While we're at it, well put together everything into one big python dictionary caled moviedict. Each entry will contain the movie name, list of genre values, the normalized popularity core, and the average rating for each movie:

In [6]:
movieDict = {}
with open('anexos/Aquivos_de_C%C3%B3digo/u.item', encoding='latin-1') as f: #arquivo com os dados
    temp = ''
    for line in f: #pra cada linha dentro do arquivo
        fields = line.rstrip('\n').split('|') #separa com | 
        movieID = int(fields[0]) #pega o id do filme e coloca como int
        name = fields[1] #pega o nome do filme
        genres = fields[5:25] #pega os generos do 5 ao 25
        genres = map(int, genres) #converte pra int
        movieDict[movieID] = (name, genres, movieNormalizedNumRatings.loc[movieID].get('size'),movieProperties.loc[movieID].rating.get('mean'))
        #joga pro dicionario o filme

For example, heres the record we end up with for movie ID 1, "Toy Story"

In [29]:
movieDict[1]

('Toy Story (1995)',
 np.float64(0.7735849056603774),
 size    452.000000
 mean      3.878319
 Name: 1, dtype: float64)

Now lets define a function that computes the distance between two movies based on how similar their genres are, and how similar their popularity is. Just to make sure it works well compute the distance between movie ID 2 and 4

In [17]:
from scipy import spatial

In [30]:
def ComputeDistance(a, b): #funcao que pega a diferenca dos generos de filme e da popularidade e ve se eles sao mt diferentes 
    genreA = list(a[1])
    genreB = list(b[1])

    if len(genreA) == 0 or len(genreB) == 0:
        return 1

    genreDistance = spatial.distance.cosine(genreA, genreB) #calcula a distancia do cosseno 

    popularityA = a[2]
    popularityB = b[2]

    popularityDistance = abs(popularityA - popularityB) #abs e pra deixar sempre positivo e aq pega a distancia da popularidade dos dois

    return genreDistance + popularityDistance 


ComputeDistance(movieDict[2], movieDict[4])

np.float64(0.8004574042309892)

Remember the higher the distance, the less similar the movies are, Let chech what movies 2 and 4 actually are - and confirm they re not really all that similar:

In [31]:
print(movieDict[2])
print(movieDict[4])

('GoldenEye (1995)', <map object at 0x7f4e78669840>, np.float64(0.22298456260720412), size    131.000000
mean      3.206107
Name: 2, dtype: float64)
('Get Shorty (1995)', <map object at 0x7f4e78603e00>, np.float64(0.3567753001715266), size    209.000000
mean      3.550239
Name: 4, dtype: float64)


In [32]:
import operator

def getNeighbors(movieID, K):
    distances = []
    for movie in movieDict: #cada filme no dicionario
        if (movie != movieID): #nao pega o mesmo filme
            dist = ComputeDistance(movieDict[movieID], movieDict[movie]) #pega a distancia entre eles
            distances.append((movie, dist)) #adiciona o filme e a distancia na lista

    distances.sort(key = operator.itemgetter(1)) #ordena pela menor distancia
    neighbors = []
    for x in range(K): #repete K vezes
        neighbors.append(distances[x][0]) #na posicao x da lista pega o id do filme

    return neighbors


K = 10 
avgRating = 0
neighbors = getNeighbors(1, K)

for neighbor in neighbors:
    avgRating += movieDict[neighbor][3] #pega a nota do filme e joga pro avgrating
    print (movieDict[neighbor][0] + " " + str(movieDict[neighbor][3])) 

avgRating /= float(K) #pega o avrating e divide pela qntd de K

GoldenEye (1995) size    131.000000
mean      3.206107
Name: 2, dtype: float64
Four Rooms (1995) size    90.000000
mean     3.033333
Name: 3, dtype: float64
Get Shorty (1995) size    209.000000
mean      3.550239
Name: 4, dtype: float64
Copycat (1995) size    86.000000
mean     3.302326
Name: 5, dtype: float64
Shanghai Triad (Yao a yao yao dao waipo qiao) (1995) size    26.000000
mean     3.576923
Name: 6, dtype: float64
Twelve Monkeys (1995) size    392.000000
mean      3.798469
Name: 7, dtype: float64
Babe (1995) size    219.000000
mean      3.995434
Name: 8, dtype: float64
Dead Man Walking (1995) size    299.000000
mean      3.896321
Name: 9, dtype: float64
Richard III (1995) size    89.000000
mean     3.831461
Name: 10, dtype: float64
Seven (Se7en) (1995) size    236.000000
mean      3.847458
Name: 11, dtype: float64


In [33]:
avgRating #media dos filmes

size    177.700000
mean      3.603807
Name: 2, dtype: float64

In [34]:
movieDict[1]

('Toy Story (1995)',
 np.float64(0.7735849056603774),
 size    452.000000
 mean      3.878319
 Name: 1, dtype: float64)

# Activity

Out choice of 10 for K was arbitrary - what effect do differejt K values on the results?

Our distance metric is also somewhat arbitrary - we just took the cosine distance between the genres and added it to the ifference between the normalized popularity scores, CAn u improve on that?